[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VaishnaviJagtap18/42-days-aiml-challenge/blob/main/week5_nlp_llms/day33_tokenization_embeddings/day33_notebook.ipynb)

# Day 33 / 42: Tokenization and Embeddings
### #42DaysOfML | Week 5: NLP and LLMs

**Resources used to build this notebook:**
- [HuggingFace NLP Course](https://huggingface.co/learn/nlp-course) — tokenization concepts
- [mlabonne/llm-course](https://github.com/mlabonne/llm-course) — LLM fundamentals track
- [sentence-transformers docs](https://www.sbert.net) — embedding and similarity code

---

## What You'll Learn
1. Why text must be converted to numbers before any model can process it
2. Three tokenization strategies — character, word, subword — with real tradeoffs
3. BPE (Byte Pair Encoding) from scratch — the exact algorithm GPT-4 uses
4. What embeddings are and why they capture meaning, not just identity
5. Word2Vec analogy arithmetic: `king - man + woman = queen`
6. Semantic search: find the most similar sentence to a query using cosine similarity
7. Real sentence embeddings with `sentence-transformers`

---

In [ ]:
# Install required libraries
# sentence-transformers covers both embedding generation and cosine similarity
!pip install sentence-transformers matplotlib numpy --quiet

## Section 1: Why Tokenization Exists

Every neural network is a mathematical function. It takes numbers as input and produces numbers as output. Raw text is not numbers.

Tokenization is the step that converts text into a sequence of integer IDs a model can process. The choices made here have consequences that ripple through everything: how well the model handles rare words, how long a sequence it can process, how many tokens a given sentence costs (which maps directly to API costs).

Three strategies exist, and each one is a different tradeoff:

| Strategy | How it splits | Problem it solves | Problem it creates |
|---|---|---|---|
| **Character-level** | Every character is a token | No unknown words ever | Very long sequences, no word-level meaning |
| **Word-level** | Every word is a token | Captures word meaning | OOV problem: unseen words become `[UNK]` |
| **Subword (BPE)** | Common sequences are merged | Handles rare words by splitting into known parts | Slightly less intuitive |

GPT-4, LLaMA, BERT, and virtually every production LLM today uses subword tokenization. The specific algorithm is Byte Pair Encoding (BPE).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ----------------------------------------------------------------
# Three tokenization strategies on the same sentence
# ----------------------------------------------------------------

sentence = "The transformer architecture changed NLP in 2017"

# Strategy 1: Character-level
char_tokens = list(sentence)

# Strategy 2: Word-level
word_tokens = sentence.lower().split()

# Strategy 3: Subword — approximation of what GPT tokenizers produce
# Real BPE output for this sentence (from tiktoken cl100k_base)
subword_tokens = ['The', ' transformer', ' architecture', ' changed', ' NLP', ' in', ' 2017']

print("=" * 60)
print(f"Original: '{sentence}'")
print("=" * 60)
print(f"\n1. Character-level ({len(char_tokens)} tokens):")
print(f"   {char_tokens}")
print(f"\n2. Word-level ({len(word_tokens)} tokens):")
print(f"   {word_tokens}")
print(f"\n3. Subword/BPE ({len(subword_tokens)} tokens):")
print(f"   {subword_tokens}")

# ----------------------------------------------------------------
# OOV (Out-Of-Vocabulary) problem with word tokenizers
# ----------------------------------------------------------------
print("\n" + "=" * 60)
print("OOV problem with word tokenizers:")
print("=" * 60)

# If your vocab was built in 2020, these words didn't exist yet
unseen_words = ['ChatGPT', 'hallucination', 'tokenization', 'multimodal', 'Sora']
training_vocab = {'the', 'cat', 'sat', 'mat', 'on', 'deep', 'learning', 'model', 'neural'}

print("\nWords a 2020-era word tokenizer might not have seen:")
for w in unseen_words:
    status = 'IN VOCAB' if w.lower() in training_vocab else '[UNK] — model sees nothing useful'
    print(f"  {w:20s}: {status}")

print("\nWith BPE, 'ChatGPT' splits into ['Chat', 'G', 'PT'] — still meaningful subwords.")
print("With word tokenizer, it becomes [UNK] — zero information.")

In [ ]:
# ----------------------------------------------------------------
# Visualise: token count comparison across strategies
# ----------------------------------------------------------------
strategies = ['Character\n(55 tokens)', 'Word\n(7 tokens)', 'BPE Subword\n(7 tokens)']
counts = [len(char_tokens), len(word_tokens), len(subword_tokens)]
colors = ['#F44336', '#FF9800', '#4CAF50']
notes = ['Too long,\nno word meaning', 'Good length,\nOOV problem', 'Best of both\n(GPT-4 uses this)']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(strategies, counts, color=colors, edgecolor='black', alpha=0.85, width=0.5)
for bar, count, note in zip(bars, counts, notes):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{count} tokens', ha='center', fontsize=12, fontweight='bold')
    ax.text(bar.get_x() + bar.get_width()/2, -6,
            note, ha='center', fontsize=9, style='italic', color='#444')

ax.set_ylabel('Token Count', fontsize=12)
ax.set_title(f'Tokenization Strategies on: "{sentence}"', fontsize=13, fontweight='bold')
ax.set_ylim(-10, 65)
ax.grid(True, alpha=0.3, axis='y')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

## Section 2: BPE (Byte Pair Encoding) From Scratch

BPE starts with characters and iteratively merges the most frequent adjacent pair into a single token. After enough merges, common words are single tokens. Rare words get split into smaller known pieces.

**The algorithm in 3 steps:**
1. Start: split every word into characters + an end-of-word marker `</w>`
2. Count: find the most frequent adjacent pair across the entire corpus
3. Merge: replace every occurrence of that pair with a new combined token
4. Repeat steps 2-3 for N merges (GPT-2 uses 50,000 merges)

This is the exact algorithm described in the paper "Neural Machine Translation of Rare Words with Subword Units" (Sennrich et al., 2016), which kickstarted modern tokenization.

In [ ]:
from collections import Counter
import re

# ----------------------------------------------------------------
# BPE algorithm from scratch
# This is what tiktoken, HuggingFace tokenizers, and SentencePiece
# all implement under the hood (with C++ optimisations for speed)
# ----------------------------------------------------------------

def get_vocab(text: str) -> dict:
    """Split each word into characters + end-of-word marker."""
    vocab = Counter()
    for word in text.split():
        # 'low' becomes 'l o w </w>'
        vocab[' '.join(list(word)) + ' </w>'] += 1
    return dict(vocab)

def get_pair_stats(vocab: dict) -> Counter:
    """Count frequency of every adjacent pair across all words."""
    pairs = Counter()
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i+1])] += freq
    return pairs

def merge_vocab(pair: tuple, vocab: dict) -> dict:
    """Replace every occurrence of `pair` with its merged form."""
    new_vocab = {}
    bigram = re.escape(' '.join(pair))
    pattern = re.compile(r'(?<![\S])' + bigram + r'(?![\S])')
    for word in vocab:
        new_word = pattern.sub(''.join(pair), word)
        new_vocab[new_word] = vocab[word]
    return new_vocab


# ---- Run BPE on a small corpus ----
corpus = 'low lower newest widest lowest newer low lower new widest'

vocab = get_vocab(corpus)
print("INITIAL VOCAB (characters + </w> marker):")
for word, freq in sorted(vocab.items(), key=lambda x: -x[1]):
    print(f"  '{word}' x{freq}")

print("\n" + "=" * 55)
print("BPE MERGES (10 iterations):")
print("=" * 55)

merge_history = []
for i in range(10):
    pairs = get_pair_stats(vocab)
    if not pairs:
        break
    best_pair = max(pairs, key=pairs.get)
    freq = pairs[best_pair]
    vocab = merge_vocab(best_pair, vocab)
    merge_history.append((best_pair, freq))
    print(f"  Merge {i+1:2d}: '{best_pair[0]}' + '{best_pair[1]}' "
          f"-> '{best_pair[0]+best_pair[1]}' (appeared {freq}x)")

print("\nFINAL VOCAB after 10 merges:")
for word, freq in sorted(vocab.items(), key=lambda x: -x[1]):
    print(f"  '{word}' x{freq}")

In [ ]:
# ----------------------------------------------------------------
# Visualise BPE merge sequence
# ----------------------------------------------------------------
fig, ax = plt.subplots(figsize=(12, 3))

for i, (pair, freq) in enumerate(merge_history):
    merged = pair[0] + pair[1]
    ax.text(i, 0.6, f"{pair[0]}+{pair[1]}",
            ha='center', va='center', fontsize=8.5,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#D5E8F0', edgecolor='#0066CC', linewidth=1.5))
    ax.text(i, 0.3, f"-> '{merged}'",
            ha='center', va='center', fontsize=8, color='#0066CC', fontweight='bold')
    ax.text(i, 0.1, f"freq={freq}",
            ha='center', va='center', fontsize=7.5, color='#555')

ax.set_xlim(-0.7, len(merge_history) - 0.3)
ax.set_ylim(0, 1)
ax.axis('off')
ax.set_title('BPE Merge Sequence: each box = one merge operation', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

print("Key insight: BPE always merges the MOST FREQUENT pair first.")
print("Common sequences (like 'lo' in low/lower/lowest) get merged early.")
print("Rare character combos stay as individual characters — handled but not wasted on a token ID.")

## Section 3: What Embeddings Actually Are

After tokenization, each token is a number (an ID). But token ID 4521 for 'king' and ID 4522 for 'queen' are just two integers — they tell the model nothing about the relationship between the words.

**Embeddings fix this.** An embedding maps each token ID to a dense vector of real numbers (e.g. 768 numbers for BERT, 1536 for OpenAI's `text-embedding-3-small`). These vectors are *learned* during training. Words that appear in similar contexts end up with similar vectors.

**Why similarity in vector space = similarity in meaning:**
During training on billions of sentences, 'king' and 'queen' both appear near words like 'throne', 'crown', 'royal', 'palace'. Their vectors get pulled toward similar regions of the embedding space. Words that never appear together end up far apart.

**Cosine similarity** measures the angle between two vectors, not their magnitude:
```
cosine_similarity(a, b) = (a · b) / (||a|| × ||b||)
```
Range: -1 (opposite) to 0 (unrelated) to 1 (identical direction).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# ----------------------------------------------------------------
# Toy Word2Vec-style embeddings to demonstrate the math
# In production, these 4 numbers would be 300 (Word2Vec) or 768 (BERT)
# Dims: [royalty, gender_male, geography_france, geography_uk]
# ----------------------------------------------------------------

embeddings = {
    'king':    np.array([0.92, 0.85, 0.02, 0.01]),
    'queen':   np.array([0.91, -0.82, 0.02, 0.01]),
    'man':     np.array([0.05, 0.88, 0.01, 0.01]),
    'woman':   np.array([0.05, -0.86, 0.01, 0.01]),
    'paris':   np.array([0.01, 0.01, 0.97, 0.05]),
    'france':  np.array([0.01, 0.01, 0.95, 0.06]),
    'london':  np.array([0.01, 0.01, 0.05, 0.96]),
    'england': np.array([0.01, 0.01, 0.06, 0.95]),
    'dog':     np.array([0.01, 0.01, 0.01, 0.01]),
    'cat':     np.array([0.02, 0.01, 0.01, 0.01]),
}

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


# ---- 1. Basic similarity pairs ----
print("WORD SIMILARITY (cosine):")
pairs = [
    ('king', 'queen', 'similar royalty'),
    ('paris', 'france', 'city and country'),
    ('paris', 'london', 'both capitals, different countries'),
    ('king', 'paris', 'completely unrelated'),
    ('dog', 'cat', 'both animals but no royalty/geo signal'),
]
for w1, w2, note in pairs:
    sim = cosine_similarity(embeddings[w1], embeddings[w2])
    bar = '|' * int(abs(sim) * 20)
    print(f"  {w1:8s} <-> {w2:8s}: {sim:+.4f}  {bar}  ({note})")

# ---- 2. Famous word analogy: king - man + woman = ? ----
print("\n" + "=" * 55)
print("WORD ANALOGY: king - man + woman = ?")
print("=" * 55)

result_vec = embeddings['king'] - embeddings['man'] + embeddings['woman']

# Find the closest word (excluding 'king', 'man', 'woman')
exclude = {'king', 'man', 'woman'}
scores = {
    word: cosine_similarity(result_vec, vec)
    for word, vec in embeddings.items()
    if word not in exclude
}
ranked = sorted(scores.items(), key=lambda x: -x[1])

print(f"\nResult vector closest to:")
for word, score in ranked:
    arrow = " <-- ANSWER" if word == ranked[0][0] else ""
    print(f"  {word:10s}: {score:.4f}{arrow}")

In [ ]:
# ----------------------------------------------------------------
# Full similarity matrix heatmap
# ----------------------------------------------------------------
words = list(embeddings.keys())
n = len(words)
sim_matrix = np.zeros((n, n))
for i, w1 in enumerate(words):
    for j, w2 in enumerate(words):
        sim_matrix[i, j] = cosine_similarity(embeddings[w1], embeddings[w2])

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(sim_matrix, cmap='Blues', vmin=-0.2, vmax=1.0)

ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(words, rotation=45, ha='right', fontsize=11)
ax.set_yticklabels(words, fontsize=11)

for i in range(n):
    for j in range(n):
        val = sim_matrix[i, j]
        color = 'white' if val > 0.6 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                fontsize=8.5, color=color)

plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title('Cosine Similarity Matrix\nDark blue = semantically similar', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Clusters you should see in the heatmap:")
print("  Top-left block:  king, queen, man, woman  (royalty/gender cluster)")
print("  Middle block:    paris, france              (French geography cluster)")
print("  Right block:     london, england            (UK geography cluster)")
print("  Bottom-right:    dog, cat                   (animals, low similarity to everything else)")

## Section 4: From Word Embeddings to Sentence Embeddings

Word embeddings capture word-level meaning. But you usually need to compare *sentences*, not individual words. "The dog chased the cat" and "A feline was pursued by a canine" mean the same thing. Their word embeddings are different; a good sentence embedding should place them close together.

**Sentence-transformers** (from the paper "Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks", Reimers & Gurevych 2019) produces dense sentence embeddings trained specifically to capture sentence-level meaning. `all-MiniLM-L6-v2` is the standard lightweight choice: 22M parameters, 384 dimensions, fast enough to embed millions of sentences on a laptop.

This is what powers semantic search inside Notion, Obsidian AI, and most document retrieval products you use today.

In [ ]:
from sentence_transformers import SentenceTransformer, util
import numpy as np

# Downloads ~80MB on first run, cached after that
model = SentenceTransformer('all-MiniLM-L6-v2')

print(f"Model: all-MiniLM-L6-v2")
print(f"Embedding dimensions: 384")
print(f"Parameters: ~22M")
print("=" * 55)

# ---- Test 1: Semantic similarity between sentence pairs ----
pairs = [
    ("The dog chased the cat.",
     "A feline was pursued by a canine.",
     "Same meaning, completely different words"),

    ("Gradient descent minimizes the loss function.",
     "The optimizer reduces prediction error step by step.",
     "Same concept, different phrasing"),

    ("The weather is nice today.",
     "Neural networks have multiple hidden layers.",
     "Completely unrelated sentences"),

    ("Backpropagation computes gradients using the chain rule.",
     "Backpropagation computes gradients using the chain rule.",
     "Identical sentences"),
]

print("\nSENTENCE SIMILARITY (real embeddings):")
for s1, s2, note in pairs:
    e1 = model.encode(s1, convert_to_tensor=True)
    e2 = model.encode(s2, convert_to_tensor=True)
    sim = float(util.cos_sim(e1, e2))
    bar = '|' * int(sim * 25)
    print(f"\n  Pair: {note}")
    print(f"  S1: '{s1[:60]}'")
    print(f"  S2: '{s2[:60]}'")
    print(f"  Cosine similarity: {sim:.4f}  {bar}")

In [ ]:
# ----------------------------------------------------------------
# SECTION 5: Semantic Search
# Find the most relevant sentences in a corpus given a query
# This is the core of every document retrieval and RAG system
# ----------------------------------------------------------------
import time

corpus = [
    "Backpropagation uses the chain rule to compute gradients in neural networks.",
    "The Adam optimizer combines momentum and adaptive learning rates.",
    "Transfer learning reuses weights from a model trained on a large dataset.",
    "Convolutional neural networks process spatial data through learned filters.",
    "RAG retrieves relevant documents before generating an LLM response.",
    "Attention mechanisms let models focus on relevant parts of the input sequence.",
    "K-Means clustering groups data points by minimizing within-cluster variance.",
    "Random forests reduce variance by averaging predictions across many trees.",
    "Fine-tuning updates pretrained model weights on a smaller task-specific dataset.",
    "Vector databases store embeddings and support fast approximate nearest neighbor search.",
    "Tokenization converts raw text into integer IDs a model can process.",
    "BERT is a bidirectional encoder that reads context from both directions simultaneously.",
]

# Encode the full corpus once and cache — this is critical for production efficiency
t0 = time.time()
corpus_embeddings = model.encode(corpus, convert_to_tensor=True)
encode_time = time.time() - t0
print(f"Encoded {len(corpus)} sentences in {encode_time:.3f}s")
print(f"Corpus embeddings shape: {corpus_embeddings.shape}  (sentences x dimensions)")


def semantic_search(query: str, corpus: list, corpus_embeddings, top_k: int = 3):
    query_embedding = model.encode(query, convert_to_tensor=True)
    scores = util.cos_sim(query_embedding, corpus_embeddings)[0]
    top_results = scores.topk(k=top_k)
    return [(corpus[idx], float(score))
            for score, idx in zip(top_results.values, top_results.indices)]


queries = [
    "How do neural networks learn from data?",
    "What is used to find similar documents quickly?",
    "How does fine-tuning differ from training from scratch?",
]

print("\n" + "=" * 60)
print("SEMANTIC SEARCH RESULTS")
print("=" * 60)

for query in queries:
    print(f"\nQuery: '{query}'")
    print("-" * 50)
    results = semantic_search(query, corpus, corpus_embeddings)
    for rank, (sentence, score) in enumerate(results, 1):
        print(f"  #{rank} [{score:.4f}] {sentence}")

In [ ]:
# ----------------------------------------------------------------
# SECTION 6: API embedding cost — a real production concern
# OpenAI text-embedding-3-small: $0.00002 per 1K tokens
# ----------------------------------------------------------------

print("EMBEDDING COST CALCULATOR (OpenAI text-embedding-3-small)")
print("=" * 55)
print(f"Price: $0.00002 per 1,000 tokens\n")

scenarios = [
    ("Small project (10K docs, avg 200 tokens)",  10_000,  200),
    ("Startup knowledge base (100K docs, 500t)",  100_000, 500),
    ("Enterprise codebase (1M files, 800t)",      1_000_000, 800),
]

price_per_1k = 0.00002

for label, n_docs, avg_tokens in scenarios:
    total_tokens = n_docs * avg_tokens
    cost_once = (total_tokens / 1000) * price_per_1k
    cost_monthly_reruns = cost_once * 4  # re-embed monthly with model updates
    print(f"{label}")
    print(f"  Total tokens:         {total_tokens:>15,}")
    print(f"  One-time embed cost:  ${cost_once:>14.2f}")
    print(f"  If re-embedded 4x/yr: ${cost_monthly_reruns:>14.2f}")
    print()

print("Production fix: cache embeddings in a vector database.")
print("Only re-embed documents that actually changed.")
print("A startup with 100K documents should re-embed <1% of them per week.")
print("That drops 4x/year cost from ~$4 to ~$0.04/week.")

In [ ]:
# ----------------------------------------------------------------
# Visualise actual embedding vectors (first 50 dims of 384)
# Shows that similar sentences produce similar activation patterns
# ----------------------------------------------------------------
sentences_to_compare = [
    "Tokenization converts text into integer IDs.",
    "Text must be split into tokens before a model can process it.",
    "The stock market fell 3% on Tuesday.",
]

embs = model.encode(sentences_to_compare)

fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True)
for i, (ax, sentence, emb) in enumerate(zip(axes, sentences_to_compare, embs)):
    ax.bar(range(50), emb[:50],
           color='#0066CC' if i < 2 else '#F44336', alpha=0.7, width=0.8)
    ax.set_ylabel('Value', fontsize=9)
    label = f"S{i+1}: '{sentence[:55]}{'...' if len(sentence)>55 else ''}'"
    ax.set_title(label, fontsize=9, fontweight='bold')
    ax.axhline(y=0, color='black', linewidth=0.5)
    ax.grid(True, alpha=0.2, axis='y')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

axes[-1].set_xlabel('Embedding dimension (first 50 of 384)', fontsize=10)
fig.suptitle('Embedding Vectors: S1 and S2 (same meaning, blue) vs S3 (unrelated, red)\n'
             'S1 and S2 should show similar activation patterns; S3 should look different',
             fontsize=11, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# Quantify visually
s1, s2, s3 = embs
sim_12 = float(util.cos_sim(s1, s2))
sim_13 = float(util.cos_sim(s1, s3))
print(f"\nS1 vs S2 (same meaning):   {sim_12:.4f}")
print(f"S1 vs S3 (different topic): {sim_13:.4f}")
print(f"\nThe activation patterns confirm what cosine similarity tells you.")

## Real World Problem: Re-embedding Everything on Model Upgrades

In November 2023, OpenAI released `text-embedding-3-small`, replacing `text-embedding-ada-002`. The new model produces 1536-dimensional embeddings instead of 1024. Any company that had stored `ada-002` embeddings in a vector database had a problem: the old embeddings are not compatible with the new model's vector space. You can't compare a `ada-002` embedding against a `3-small` embedding.

A startup with 500,000 documents embedded with `ada-002` faced this choice:
- Re-embed everything with `3-small`: cost ~$5 in API fees, but required spinning up a batch job, rewriting their vector DB schema, and redeploying.
- Stay on `ada-002`: no change required, but fall behind on retrieval quality.

**The engineers who had cached their raw document chunks separately** (not just the embeddings) re-embedded in one afternoon. The engineers who had only stored embeddings (not the source text that generated them) couldn't re-embed without reconstructing their entire document pipeline from scratch.

**Lesson:** Always store the source text alongside the embedding. Embeddings are reproducible; lost source text is not.

## Interview Corner: MNC-Level Questions

---

**Q1: Why does GPT-4 use BPE tokenization instead of word-level tokenization?**

*What they're testing:* Whether you understand the OOV problem and subword tradeoffs.

*Answer direction:* Three reasons. (1) Word tokenizers fail on words not in the training vocabulary — they become `[UNK]`, discarding all information. New product names, misspellings, technical jargon, and code snippets hit this constantly. (2) Word vocabularies get large fast — English alone has 170,000+ words, and GPT-4 handles 100+ languages plus code. BPE covers all of this with ~100K tokens. (3) BPE handles any string by falling back to character-level representations for truly unknown sequences. GPT-4 can tokenize `def my_unusual_function_name_v3()` even if it never appeared in training.

---

**Q2: Two sentences with no words in common can have a cosine similarity of 0.95. How?**

*What they're testing:* Understanding of what embeddings actually encode.

*Answer direction:* Embeddings capture *distributional meaning*, not word identity. A model trained on billions of sentences learns that 'dog' and 'canine', 'purchase' and 'buy', 'died' and 'passed away' appear in similar contexts and serve similar grammatical roles. Their vectors end up close in the embedding space. So "The feline pursued the canine" and "The dog chased the cat" can have near-identical embeddings despite sharing zero tokens, because the model has learned the semantic relationships between all those word pairs.

---

**Q3: Your semantic search returns irrelevant results for short queries like 'Python'. What's wrong?**

*What they're testing:* Practical embedding failure modes.

*Answer direction:* Single-word queries are inherently ambiguous — 'Python' could mean the language, the snake, or Monty Python. The embedding vector for a single ambiguous word sits in a vague region of the space that doesn't point clearly toward any meaning cluster. Fix: (1) Use a query expansion step — ask an LLM to rewrite the short query as a full sentence before embedding it. (2) Switch to a model trained for asymmetric retrieval (where short queries match long documents) — `multi-qa-MiniLM-L6-cos-v1` from sentence-transformers is designed for exactly this. (3) Combine embedding similarity with keyword search (BM25) and merge scores — this is called hybrid search and is what Elasticsearch and Pinecone both recommend.

---

**Q4: What is the difference between token embeddings and position embeddings in a Transformer?**

*What they're testing:* Attention to input representation detail.

*Answer direction:* Token embeddings encode *what* each token is — they map a token ID to a learned dense vector. Position embeddings encode *where* each token sits in the sequence. Self-attention is permutation-invariant by design, meaning it treats 'cat chased dog' and 'dog chased cat' as identical without positional information. Position embeddings (either sinusoidal as in the original Transformer, or learned as in BERT) are added to token embeddings so the model knows each token's position. The final input to the first Transformer layer is `token_embedding + position_embedding`, a single vector carrying both what and where.

---

**Q5: A company re-trains their embedding model and wants to update their vector database without downtime. How?**

*What they're testing:* Production deployment thinking.

*Answer direction:* Blue-green embedding migration. Keep the old embedding index serving live traffic (blue). Spin up a new index (green) and batch re-embed all documents with the new model in the background. Once the new index is fully built and validated on a held-out query set, cut traffic from blue to green atomically. Zero downtime. Important: keep the old index available for rollback for at least 48 hours in case query quality regresses. During the migration window, any new documents that arrive must be written to both indexes simultaneously to stay in sync.

## ML Spotlight

**Matryoshka Representation Learning (MRL) — OpenAI's embedding trick**

Standard embeddings are fixed-size. OpenAI's `text-embedding-3-small` outputs 1536 dimensions and you use all of them. If you only need 256 dimensions (for a fast approximate search with lower storage cost), you'd need to train a completely separate smaller model.

MRL (Kusupati et al., 2022) solves this by training embeddings so that the *first N dimensions are themselves a valid, high-quality embedding*. A 1536-dim MRL embedding truncated to its first 256 dimensions still outperforms a dedicated 256-dim model trained from scratch.

OpenAI integrated this into `text-embedding-3-small` and `text-embedding-3-large`. You can set `dimensions=256` in the API call and get a real 256-dim embedding, not a truncated 1536-dim one.

In practice: production teams now use 256-dim embeddings for fast first-stage retrieval (high recall, low cost) and 1536-dim embeddings for re-ranking the top-100 results. This two-stage pattern is used in production at Pinecone, Weaviate, and Cohere.

Paper: [Matryoshka Representation Learning](https://arxiv.org/abs/2205.13147)

## Practice Exercise

**Task 1:** Extend the BPE implementation to encode a new word that wasn't in the training corpus. Given the merge rules learned above, apply them in order to tokenize the word `'lowest'`.

**Task 2:** Build a semantic search engine over a mini FAQ.
```python
faq = [
    "How do I reset my password?",
    "What payment methods do you accept?",
    "How long does shipping take?",
    "Can I return a product after 30 days?",
    "How do I contact customer support?",
]
# User query: "I forgot my login credentials"
# Expected top result: "How do I reset my password?"
```
Encode the FAQ, encode the query, compute cosine similarities, return the top match.

**Task 3:** Run this comparison:
```python
# Embed these two sentences with all-MiniLM-L6-v2
s1 = "I love programming in Python."
s2 = "Python is my favourite programming language."
s3 = "The Burmese python is the largest snake in Southeast Asia."
```
Which two are most similar? Does the model correctly disambiguate 'Python'?

---

**What's Next**

Day 34: Fine-tuning vs Prompting — when a well-crafted prompt beats a fine-tuned model, when it doesn't, and how to measure the difference before spending money on either.